# Models  模型
## Parameters  参数
- model 模型 必填
- api_key
- temperature  温度
- max_tokens  最大令牌数
- timeout 最长响应时间
- max_retries  最大重试次数
 > 如果请求因网络超时或速率限制等问题而失败，系统将尝试重新发送请求的最大次数。重试采用带抖动的指数退避算法。网络错误、速率限制错误（429）和服务器错误（5xx）会自动重试。客户端错误（例如 401（未授权）或 404）不会重试。对于在不稳定网络上长时间运行的代理任务，建议将此值增加到 10-15。


##  Invocation  调用
必须调用聊天模型才能生成输出。有三种主要的调用方法，每种方法都适用于不同的使用场景。
### Invoke  调用
传入一条消息或一条消息列表。

聊天模型可以接收消息列表来表示对话历史记录。每条消息都有一个角色，模型使用该角色来指示对话中消息的发送者。
```Dictionary format  字典格式
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]
```
```Message objects  消息对象
conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications.")
]
```
> 如果您的调用返回类型为字符串，请确保您使用的是聊天模型而不是 LLM。传统的文本补全 LLM 直接返回字符串。LangChain 聊天模型以“Chat”为前缀，例如 ChatOpenAI (/oss/integrations/chat/openai)。
### Stream  溪流
大多数模型都能在生成输出内容的同时进行流式传输。通过逐步显示输出，流式传输显著提升了用户体验，尤其是在处理较长的响应时。

调用 `stream()` 返回一个`iterator`(迭代器) ,它会在生成过程中实时输出数据块。您可以使用循环来实时处理每个数据块：

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from dotenv import load_dotenv
import os

load_dotenv()

base_url = os.getenv("GLM_BASE_URL")
api_key = os.getenv("GLM_API_KEY")

model = ChatOpenAI(
    base_url=base_url,
    api_key=api_key,
    model = "glm-4.5-air",
    streaming=True,
    max_tokens=128,
    # reasoning={
    #     "effort":"medium"
    # }
)

In [ ]:

agent = create_agent(
    model=model,
)

# for chunk in model.stream("Hello"):
#     print(chunk.content, end=" ", flush=True)

# 流式工具调用、推理和其他内容
for chunk in model.stream("你好"):
    for block in chunk.content_blocks:
        if block["type"] == "reasoning" and (reasoning := block.get("reasoning")):
            print(f"Reasoning: {reasoning}")
        elif block["type"] == "tool_call_chunk":
            print(f"Tool call chunk: {block}")
        elif block["type"] == "text":
            print(block["text"], end="", flush=True)


# 基本文本流
# for chunk in agent.stream(
#     {
#         "messages": [
#             {"role": "user", "content": "你好"}
#         ]
#     },
#     stream_mode="messages"
# ):
#     msg = chunk[0]
#     if msg.content:
#         print(msg.content, end="", flush=True)

与 invoke() 函数在模型生成完整响应后返回单个 AIMessage 不同， stream() 返回多个 AIMessageChunk 对象，每个对象包含输出文本的一部分。重要的是，stream 中的每个块都旨在通过求和的方式组合成一条完整的消息：

In [ ]:
full = None  # None | AIMessageChunk
for chunk in model.stream("What color is the sky?"):
    full = chunk if full is None else full + chunk
    print(full.text, end="", flush=True)

# The
# The sky
# The sky is
# The sky is typically
# The sky is typically blue
# ...

print(full.content_blocks)
# [{"type": "text", "text": "The sky is typically blue..."}]

> 只有当程序中的所有步骤都知道如何处理数据块流时，流式传输才能正常工作。例如，如果一个应用程序需要先将整个输出存储在内存中才能进行处理，那么它就不具备流式传输能力。

### Advanced streaming topics 高级流媒体主题
#### Streaming events  流媒体活动
LangChain 聊天模型还可以使用 `astream_events()` 流式传输语义事件。

这样可以简化基于事件类型和其他元数据的筛选，并在后台聚合完整消息。

In [ ]:
async for event in model.astream_events("Hello"):

    if event["event"] == "on_chat_model_start":
        print(f"Input: {event['data']['input']}")

    elif event["event"] == "on_chat_model_stream":
        print(f"Token: {event['data']['chunk'].text}")

    elif event["event"] == "on_chat_model_end":
        print(f"Full message: {event['data']['output'].text}")

    else:
        pass

#### "Auto-streaming" chat models “自动流式”聊天模式
LangChain 通过在某些情况下自动启用流式模式来简化聊天模型中的流式传输，即使您没有显式调用流式方法。

回调事件允许 LangGraph stream() 和 astream_events() 实时显示聊天模型的输出。

### Batch  批
将一系列独立的模型请求批量处理，可以显著提高性能并降低成本，因为可以并行处理这些请求：

In [ ]:
responses = model.batch([
    "你好,你是蜡笔小新吗?",
    "你好,我是小新",
    "hi"
])
for response in responses:
    print(response)

默认情况下， batch() 只会返回整个批次的最终输出。如果您希望在每个输入生成完成后立即接收其输出，可以使用 batch_as_completed() 来流式传输结果：

In [ ]:
for response in model.batch_as_completed([
    "你好,你是蜡笔小新吗?",
    "你好,我是小新",
    "hi"
]):
    print(response)

>  使用 batch_as_completed() 时，结果可能乱序到达。每个结​​果都包含输入索引，以便根据需要进行匹配，从而重建原始顺序。

> 使用 batch() 或 batch_as_completed() 处理大量输入时，您可能需要控制最大并行调用次数。这可以通过设置 RunnableConfig 字典中的 max_concurrency 属性来实现。
```python
model.batch(
    list_of_inputs,
    config={
        'max_concurrency': 5,  # Limit to 5 parallel calls
    }
)
```

## Tool calling  工具调用
模型可以请求调用工具来执行诸如从数据库获取数据、搜索网络或运行代码等任务。工具是以下各项的组合：
1. `schema`模式，包括工具名称、描述和/或参数定义（通常是 JSON 模式）
2. 函数或协程执行。

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."


model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

绑定用户自定义工具时，模型的响应包含执行该工具的请求 。如果模型与代理分开使用，则需要您自行执行请求的工具并将结果返回给模型以供后续推理使用。如果使用代理 ，代理循环将自动处理工具执行循环。

### Tool execution loop  工具执行循环
当模型返回工具调用时，您需要执行这些工具并将结果返回给模型。这会创建一个对话循环，模型可以使用工具结果生成最终响应。
### Forcing tool calls  强制工具调用
强制使用工具(不指定)
`model_with_tools = model.bind_tools([tool_1], tool_choice="any")`
强制使用指定工具
`model_with_tools = model.bind_tools([tool_1], tool_choice="tool_1")`
### Parallel tool calls  并行工具调用

In [ ]:
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke(
    "What's the weather in Boston and Tokyo?"
)


# The model may generate multiple tool calls
print(response.tool_calls)
# [
#   {'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_1'},
#   {'name': 'get_weather', 'args': {'location': 'Tokyo'}, 'id': 'call_2'},
# ]


# Execute all tools (can be done in parallel with async)
results = []
for tool_call in response.tool_calls:
    if tool_call['name'] == 'get_weather':
        result = get_weather.invoke(tool_call)
    ...
    results.append(result)
results

In [ ]:
model.bind_tools([get_weather], parallel_tool_calls=False)# 关闭工具调用的并行执行

### Streaming tool calls  流式工具调用
在流式响应中，工具调用通过 ToolCallChunk 逐步构建。这样，您就可以在工具调用生成过程中看到它们，而无需等待完整的响应。

In [ ]:
for chunk in model_with_tools.stream(
    "What's the weather in Boston and Tokyo?"
):
    # Tool call chunks arrive progressively
    for tool_chunk in chunk.tool_call_chunks:
        if name := tool_chunk.get("name"):
            print(f"Tool: {name}",flush=True)
        if id_ := tool_chunk.get("id"):
            print(f"ID: {id_}",flush=True)
        if args := tool_chunk.get("args"):
            print(f"Args: {args}",flush=True)


## Structured output  结构化输出

In [ ]:
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

load_dotenv()

qwen_base_url = os.getenv("QWEN_BASE_URL")
qwen_api_key = os.getenv("QWEN_API_KEY")

print(qwen_base_url, qwen_api_key)

qwen_model = ChatOpenAI(
    base_url=qwen_base_url,
    api_key=qwen_api_key,
    model = "glm-4.7",
    streaming=True,
    max_tokens=128,
    # reasoning={
    #     "effort": "low"
    # }
)

# class Movie(BaseModel):
#     """A movie with details."""
#     title: str = Field(description="The title of the movie")
#     year: int = Field(description="The year the movie was released")
#     director: str = Field(description="The director of the movie")
#     rating: float = Field(description="The movie's rating out of 10")

# model_with_structure = qwen_model.with_structured_output(Movie)
# response = model_with_structure.invoke("Provide details about the movie Inception")
# response = qwen_model.invoke("你好")
print(response)  # Movie(title="Inception", year=2010, director="Christopher Nolan", rating=8.8)

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_structure = qwen_model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}

In [ ]:
import json

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

model_with_structure = qwen_model.with_structured_output(
    json_schema,
    method="json_schema",
)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, ...}

**结构化输出的关键考虑因素**
**方法参数** ：某些提供商支持不同的结构化输出方法：
- `json_schema` ：使用提供商提供的专用结构化输出功能。
- `function_calling` ：通过强制执行遵循给定模式的工具调用来导出结构化输出。
- `json_mode` ：某些提供商提供的 'json_schema' 的前身。它会生成有效的 JSON，但必须在提示符中描述模式。
包含原始数据 ：设置 include_raw=True 可同时获取解析后的输出和原始 AI 消息。

验证 ： TypedDict 模型提供自动验证。TypedDict 和 JSON Schema 需要手动验证。

## Advanced topics  高级主题
### Reasoning  推理
如果底层模型支持， 你可以将这个推理过程展现出来，以便更好地理解模型是如何得出最终答案的。

In [ ]:
for chunk in model.stream("你好,为什么地球是圆的?"):
    reasoning_steps = [r for r in chunk.content_blocks if r["type"] == "reasoning"]
    print(reasoning_steps if reasoning_steps else chunk.text,end="", flush=True)

### Prompt caching  提示缓存
许多服务提供商提供即时缓存功能，以减少重复处理相同令牌时的延迟和成本。这些功能可以是隐式的 ，也可以是显式的 ：
- 隐式提示缓存： 如果请求命中缓存，服务提供商会自动将节省的成本传递给目标用户。例如： OpenAI 和 Gemini 。
- 显式缓存： 服务提供商允许您手动指定缓存点，以便更好地控制缓存或确保节省成本。例如：
  - ChatOpenAI （通过 prompt_cache_key ）
  - AnthropicPromptCachingMiddleware
  - Gemini.  双子座 。
  - AWS Bedrock
> 提示缓存通常仅在输入令牌数量超过最低阈值时才会启用。
### Server-side tool use  服务器端工具的使用
一些提供商支持服务器端工具调用循环：模型可以与网络搜索、代码解释器和其他工具进行交互，并在一次对话中分析结果。
### Rate limiting  限速
为了帮助管理速率限制，聊天模型集成接受一个 rate_limiter 参数，可以在初始化期间提供该参数来控制发出请求的速率。

LangChain 内置了一个（可选的） InMemoryRateLimiter 。该限制器是线程安全的，并且可以由同一进程中的多个线程共享。


In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # 每10秒1次请求
    check_every_n_seconds=0.1,  # 检查速率限制的频率
    max_bucket_size=10,  # 控制最大突发请求数
)

qwen_model = ChatOpenAI(
    base_url=qwen_base_url,
    api_key=qwen_api_key,
    model = "glm-4.7",
    streaming=True,
    max_tokens=128,
    rate_limiter=rate_limiter#提供的速率限制器只能限制单位时间内的请求数量。如果还需要根据请求的大小进行限制，则它无法提供帮助。
)

### Log probabilities  对数概率
某些模型可以通过在初始化模型时设置 logprobs 参数来配置为返回表示给定标记可能性的标记级日志概率：

In [ ]:
qwen_model = ChatOpenAI(
    base_url=qwen_base_url,
    api_key=qwen_api_key,
    model = "glm-4.7",
    streaming=True,
    max_tokens=128
)

response = qwen_model.invoke("Why do parrots talk?")
print(response.response_metadata["logprobs"])

### Token usage  代币使用
许多模型提供程序会在调用响应中返回令牌使用信息。如果可用，此信息将包含在相应模型生成的 AIMessage 对象中。
可以使用回调或上下文管理器来跟踪应用程序中各个模型的聚合令牌计数，如下所示：

In [ ]:
#Callback handler  回调处理程序

from langchain.chat_models import init_chat_model
from langchain_core.callbacks import UsageMetadataCallbackHandler

callback = UsageMetadataCallbackHandler()
result_1 = model.invoke("你好", config={"callbacks": [callback]})
result_2 = qwen_model.invoke("你好", config={"callbacks": [callback]})
print(result_1)

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import get_usage_metadata_callback

with get_usage_metadata_callback() as cb:
    model.invoke("Hello")
    model.invoke("Hello")
    print(cb.usage_metadata)

### Invocation config  调用配置
调用模型时，可以使用 RunnableConfig 字典通过 config 参数传递额外的配置。这可以实现对执行行为、回调和元数据跟踪的运行时控制。
```
response = model.invoke(
    "Tell me a joke",
    config={
        "run_name": "joke_generation",      # Custom name for this run
        "tags": ["humor", "demo"],          # Tags for categorization
        "metadata": {"user_id": "123"},     # Custom metadata
        "callbacks": [my_callback_handler], # Callback handlers
    }
)
```
关键配置属性:
- run_name  运行名称 string (在日志和跟踪记录中标识此次特定调用。不会被子调用继承。)
- tags  标签 string[] (用于调试工具中过滤和组织的所有子调用所继承的标签。)
- metadata  元数据 object (自定义键值对，用于跟踪附加上下文，所有子调用均可继承。)
- max_concurrency  最大并发 number (控制使用 batch() 或 batch_as_completed() 时的最大并行调用次数。)
- callbacks  回调 array (用于监控和响应执行过程中事件的处理程序。)
- recursion_limit  递归限制 number (链的最大递归深度，以防止复杂管道中出现无限循环。)